# Act 4 — The Agent That Writes Its Own Tools

This is the "updates its own source code" idea from the outline, made safe and demoable: instead of editing our framework live, the agent writes a **new tool file to disk**. We load it, and a fresh agent uses it.

In [ ]:
import sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv(override=True)

from mini_agent import Agent, write_tool, load_generated_tools
from mini_agent.tools import GENERATED_TOOLS_DIR

## First, watch it get this wrong
No tools at all, and a system prompt that discourages showing its work — a realistic constraint, plenty of production prompts ask for terse answers:

In [ ]:
agent_no_tools = Agent(instructions="Answer with just the final number. No explanation, no step-by-step reasoning, no working shown.")
answer = await agent_no_tools.run("How many vowels are in 'multi-agent architecture'?")
print("\nFINAL:", answer)

That's wrong (the correct count is 9). Counting characters reliably is a known weak spot: models see tokens, not letters. Instead of hoping it reasons carefully enough every time, let's get it a tool that counts correctly by construction. We don't have one — so let's have it write its own.

## Step 1 — ask for something it has no tool for
This time we only give it `write_tool`. No calculator, no search, no code execution — and definitely no `count_vowels`.

In [ ]:
agent = Agent(
    tools=[write_tool],
    instructions=(
        "You have no tool for counting vowels in a word. Use write_tool to create one: "
        "name='count_vowels', a function count_vowels(text: str) -> int. "
        "Then tell the user it's ready."
    ),
    max_steps=4,
)
answer = await agent.run("I need to count the vowels in 'multi-agent architecture'. Write yourself a tool for it.")
print("\nFINAL:", answer)

## Step 2 — see what it wrote

In [ ]:
print((GENERATED_TOOLS_DIR / "count_vowels.py").read_text())

## Step 3 — load it and hand it to a fresh agent

In [ ]:
new_tools = load_generated_tools()
print("loaded:", [t.__name__ for t in new_tools])

agent2 = Agent(tools=new_tools, instructions="Use your tools when relevant.")
answer2 = await agent2.run("How many vowels are in 'multi-agent architecture'? Use your tool.")
print("\nFINAL:", answer2)

Compare that to the wrong answer at the top of this notebook — same question, now correct, because the agent extended its own capability set at runtime instead of guessing. No presenter code changes in between. This is a small, safe slice of "self-modifying agents": generating new tools rather than editing its own control loop. Full self-rewriting agents are a real research direction, but come with real risks — an agent that can rewrite its own guardrails needs guardrails around *that*.

**Next:** how do we know if any of this actually works? → `05_evaluation_gaia.ipynb`.